In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
DROP TABLE IF EXISTS default.courses;
DROP TABLE IF EXISTS default.courses_sink;

In [0]:
# 1) aktive Streams stoppen (falls vorhanden)
for s in spark.streams.active:
    s.stop()

# 2. Checkpoint-Verzeichnis des Streams löschen
# Dadurch "vergisst" der Stream seine komplette Historie.
checkpoint_path = "/Volumes/workspace/default/volume/_chk/courses_vw_to_sink"
dbutils.fs.rm(checkpoint_path, recurse=True)

checkpoint_path_stateful = "/Volumes/workspace/default/volume/_checkpoint_path_stateful"
dbutils.fs.rm(checkpoint_path_stateful, recurse=True)

print("Alle Tabellen und Checkpoints wurden zurückgesetzt. Sie können den Prozess neu starten.")

# Vorbereitung: Datei einlesen und als Tabelle abspeichern

In [0]:
%python
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
json_files = [f for f in all_files if f.name.endswith(".json")]

display(json_files)

In [0]:
# JSON lesen mit automatischer Typ-Erkennung
courses_df = (spark.read
    .format("json")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/volume/courses.json"))

# Erstellt eine temporäre Sicht für SQL-Befehle in diesem Notebook
courses_df.createOrReplaceTempView("courses")

In [0]:
# 1) Courses dauerhaft im Unity Catalog speichern
# Wir nutzen den Three-Level-Namespace (catalog.schema.table)
(courses_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") # Erlaubt spätere Änderungen an der Kurs-Struktur
    .saveAsTable("workspace.default.courses"))

print("Tabelle 'courses' wurde erfolgreich im Unity Catalog erstellt/aktualisiert.")

# Structured Streaming

Read Stream von PySpark API

In [0]:
# spark.readStream ermöglicht das Abfragen einer Delta-Tabelle als 
# Streaming-Quelle und erstellt daraus ein Streaming-DataFrame.
stream_df = spark.readStream.table("workspace.default.courses") # readStream = "abonniere" das Delta Transaction-Log. Spark betrachtet die Tabelle nun als "unendliche Tabelle".
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")  # Registriert Streaming-DataFrame als "View" im SQL-Namespace der aktuellen Spark-Session (das ist jetzt also eine STREAMING View)

# Erweiterte Erläuterungen:
# spark.read. = Batch
# spark.readStream. = Stream

In [0]:
# Prüfung: Ist es wirklich ein Stream?
print(stream_df.isStreaming)

Write Stream

In [0]:
# 1. Zieltabelle (Sink) im Unity Catalog anlegen
spark.sql("""
  CREATE TABLE IF NOT EXISTS workspace.default.courses_sink
  LIKE workspace.default.courses
""")

# 2. Aus der STREAMING-View lesen (bleibt ein Streaming-DataFrame)
df = spark.table("courses_streaming_tmp_vw") 

# 3. Stream in die Delta-Tabelle schreiben (Micro-Batch Modus)
q = (
  df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True) # Dieser Trigger ist in der Free Edition zwingend
    .option("checkpointLocation",
            "/Volumes/workspace/default/volume/_chk/courses_vw_to_sink")
    .toTable("workspace.default.courses_sink")
)

# Warten, bis der Micro-Batch abgeschlossen ist
q.awaitTermination()

# 4. Ergebnis im Catalog Explorer prüfen
display(spark.table("workspace.default.courses_sink"))

In [0]:
%sql
-- Daten prüfen (variante SQL)
SELECT * FROM workspace.default.courses_sink

Daten einfügen

In [0]:
%sql
INSERT INTO workspace.default.courses VALUES ('History', 1234, 'Manual Insert', 150, 'A new course');
INSERT INTO workspace.default.courses VALUES ('History', 1235, 'Manual Insert', 150, 'Another course');

In [0]:
# Write Stream (Code-Block 13) nochmals ausführen weil availableNow=True (limitierung Free Edition).
# DANACH hier weiter gehen

In [0]:
%sql
-- Inser ist in der Quelltabelle nicht vorhanden. Grund: Cache ist nicht aktualisiert
SELECT * FROM courses

In [0]:
%sql
--- die sink-Tabelle zeigt uns die vollständigen Daten (die Stream-Wahrheit)
SELECT * FROM workspace.default.courses_sink ORDER by course_id DESC

In [0]:
%sql
-- Cache löschen um ebenfalls die Wahrheit aus der Quelltabelle zu analysieren (variante Free Edition)
-- Die Bedingung "IS NOT NULL" für den Primary Key ändert das Ergebnis nicht,
-- aber zwingt Spark zu einer neuen Ausführung. 
-- Alternative Cluster neu starten
SELECT * FROM workspace.default.courses WHERE course_id IS NOT NULL

%md
## Streaming Data Manipulations in Python

In [0]:
# Ziel-Verzeichnis für den Checkpoint der Aggregation
instructor_agg_chk = "/Volumes/workspace/default/volume/_chk/courses_by_instructor_agg_final"

# 1. Alte Ergebnistabelle löschen, damit sie neu erstellt werden kann
spark.sql("DROP TABLE IF EXISTS workspace.default.courses_by_instructor")

# 2. Checkpoint löschen. KRITISCH! Sonst wird der Stream keine Daten verarbeiten.
dbutils.fs.rm(instructor_agg_chk, recurse=True)

print("✅ Altes Ergebnis und Checkpoint gelöscht. Bereit für den neuen Lauf.")

In [0]:
import pyspark.sql.functions as F

# 1. Definiere den Stream von der Quelltabelle (Unity Catalog Standard: workspace.default)
stream_df = spark.readStream.table("workspace.default.courses")

# 2. Aggregation definieren: Gruppierung nach Kursleiter
# Spark hält hierfür einen "State" (Zwischenspeicher) im Hintergrund aufrecht.
agg_df = (
    stream_df.groupBy("instructor")
             .agg(F.count("course_id").alias("total_courses"))
)

# 3. Stream starten mit "Complete"-Modus
q = (
  agg_df.writeStream
    .format("delta")
    .outputMode("complete") # WICHTIG: Schreibt bei jedem Update die GESAMTE Ergebnistabelle neu.
    .trigger(availableNow=True)
    .option("checkpointLocation", instructor_agg_chk)
    .toTable("workspace.default.courses_by_instructor")
)

# Warten auf Abschluss des Micro-Batches
q.awaitTermination()

print("✅ Streaming-Aggregation abgeschlossen. Die Tabelle wurde vollständig aktualisiert.")

In [0]:
print("⬇️ Aktuelles Ergebnis der Aggregation:")

# Abfrage der neu erstellten Zieltabelle
display(spark.table("workspace.default.courses_by_instructor"))

Insert

In [0]:
%sql
INSERT INTO workspace.default.courses VALUES ('History', 1236, 'Manual Insert2', 180, 'A new course');
INSERT INTO workspace.default.courses VALUES ('History', 1237, 'Manual Insert2', 180, 'Another new course');

In [0]:
# Zelle mit Write Stream (Code Block 23) nochmals ausführen weil availableNow=True (limitierung Free Edition).
# DANACH hier weiter gehen

In [0]:
%sql
SELECT * FROM workspace.default.courses ORDER BY course_id DESC

In [0]:
%sql
SELECT * FROM workspace.default.courses_by_instructor order by total_courses DESC

# Streaming Data Manipulations in SQL

In [0]:
# Checkpoint löschen, um den Stream komplett zurückzusetzen
# Dies ist notwendig, weil die Quelltabelle neu erstellt wurde und eine neue ID hat.
dbutils.fs.rm(checkpoint_path_stateful, True)

In [0]:
# 1. Quelle: Wir lesen kontinuierlich aus der Haupt-Tabelle
stream_df = spark.readStream.table("workspace.default.courses")

# 2. Checkpoint: Das "Gedächtnis" für die Offsets (Wo sind wir im Stream?)
checkpoint_path = "/Volumes/workspace/default/volume/_chk/courses_append_checkpoint"

# 3. Schreiben in eine persistente Delta-Tabelle (Sink)
# Da wir keine Aggregation nutzen, ist der Modus "append" perfekt.
query = (stream_df.writeStream 
         .format("delta")               
         .outputMode("append")          
         .trigger(availableNow=True) # Verarbeitet alles Verfügbare und stoppt
         .option("checkpointLocation", checkpoint_path) 
         .toTable("workspace.default.courses_raw_sink") # <--- Zieltabelle im Katalog erstellen
        )

query.awaitTermination()
print("✅ Neue Daten wurden erfolgreich in die Delta-Tabelle übertragen.")

In [0]:
%sql
SELECT instructor, count(course_id) as total_courses 
FROM workspace.default.courses_raw_sink
GROUP BY instructor
ORDER BY total_courses DESC

In [0]:
%sql
-- Wir simulieren eine neue Buchung / einen neuen Kurs
INSERT INTO workspace.default.courses 
VALUES ('Mastering Databricks Streams', 1001, 'New Instructor3', 99.90, 'Data Engineering');

In [0]:
# Write Stream (Code-Block 32) nochmals ausführen weil availableNow=True (limitierung Free Edition).
# DANACH hier weiter gehen

# OPTIONAL FÜR VERSTÄNDNIS CHECKPOINTS: Checkpoints löschen (Code-Block 31) und nochmals ausgühren. --> Der Stream "vergisst" wo er stehengeblieben ist...

In [0]:
%sql
SELECT instructor, count(course_id) as total_courses FROM workspace.default.courses_raw_sink
GROUP BY instructor
ORDER BY total_courses DESC

In [0]:
%sql
-- Zeigt die Spaltennamen und Datentypen an
DESCRIBE courses_raw_sink;

# Aufräumen

In [0]:
import os

# 1) Alle aktiven Streams stoppen
# Wichtig: Ein laufender Stream blockiert oft Schreibzugriffe auf seine Checkpoints.
for s in spark.streams.active:
    print(f"Stoppe Stream: {s.name or s.id}")
    s.stop()

# 2) Checkpoint-Verzeichnisse im Volume löschen
# Wir löschen das "Gedächtnis", damit die Streams bei "Stunde Null" beginnen.
checkpoints = [
    "/Volumes/workspace/default/volume/_chk/courses_vw_to_sink",
    "/Volumes/workspace/default/volume/_chk/courses_raw_append_stateful",
    "/Volumes/workspace/default/volume/_chk/courses_by_instructor_agg_final"
]

for path in checkpoints:
    if os.path.exists(path): # Sauberer Check vor dem Löschen
        dbutils.fs.rm(path, recurse=True)
        print(f"Checkpoint gelöscht: {path}")

# 3) Optional: Zieltabelle bereinigen (für einen echten Kaltstart)
# spark.sql("DROP TABLE IF EXISTS main.default.courses_sink")

print("\n✅ Alle Streams gestoppt und Checkpoints zurückgesetzt. Bereit für den Neustart!")

In [0]:
%sql
-- 1. Löschen der Haupttabellen und Sinks im Unity Catalog
-- Wir nutzen 'workspace.default', um sicherzugehen, dass wir im richtigen Katalog aufräumen.

DROP TABLE IF EXISTS workspace.default.courses;
DROP TABLE IF EXISTS workspace.default.courses_sink;
DROP TABLE IF EXISTS workspace.default.courses_by_instructor;
DROP TABLE IF EXISTS workspace.default.courses_raw_sink;

-- Optional: Falls wir auch die temporäre Streaming-View löschen wollen:
DROP VIEW IF EXISTS courses_streaming_tmp_vw;